In [46]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

## 0. Setup & chargement

In [47]:
df = pd.read_csv(r"..\data\processed\final.csv")
df = df.drop(columns=['Unnamed: 0', 'birth'])
df['date'] = pd.to_datetime(df['date'])

In [48]:
df.dtypes

client_id             object
sex                   object
id_prod               object
date          datetime64[ns]
session_id            object
price                float64
categ                  int64
segment               object
age                    int64
dtype: object

In [49]:
# identification des clients BtoB
ca_par_clients = df.groupby('client_id')['price'].sum().sort_values(ascending=False).reset_index()
ca_par_clients['segment'] = np.where(ca_par_clients['price'] > 50000, 'BtoB', 'BtoC')
df['segment'] = df['client_id'].map(ca_par_clients.set_index('client_id')['segment'])

In [50]:
df_btoc = df.query("segment == 'BtoC'")
df_btoc.head()

,client_id,sex,id_prod,date,session_id,price,categ,segment,age
0,c_4410,f,1_483,2021-03-13 21:35:55.949042,s_5913,15.99,1,BtoC,54
1,c_4410,f,0_1111,2021-03-22 01:27:49.480137,s_9707,19.99,0,BtoC,54
2,c_4410,f,1_385,2021-03-22 01:40:22.782925,s_9707,25.99,1,BtoC,54
3,c_4410,f,0_1455,2021-03-22 14:29:25.189266,s_9942,8.99,0,BtoC,54
4,c_4410,f,0_1420,2021-03-22 22:31:25.825764,s_10092,11.53,0,BtoC,54


## 1. CA dans le temps

### 1.1 CA mensuel + moyenne mobile

In [51]:
# Par mois

df_ca = df_btoc.set_index('date')
df_ca = df_ca.groupby(pd.Grouper(freq='ME'))['price'].sum().to_frame()
df_ca = df_ca.rename(columns={'price' : 'ca'})
df_ca['moy_mobile'] = round(df_ca['ca'].rolling(window=3).mean(), 2)
df_ca.head()

# ------------------------

# Par jours

df_ca_p = df.set_index('date')
df_ca_p = df_ca_p.groupby(pd.Grouper(freq='D'))['price'].sum().to_frame()
df_ca_p = df_ca_p.rename(columns={'price' : 'ca'})
df_ca_p['moy_mobile_3j'] = round(df_ca_p['ca'].rolling(window=3).mean(), 2)
df_ca_p['moy_mobile_7j'] = round(df_ca_p['ca'].rolling(window=7).mean(), 2)
df_ca_p['moy_mobile_15j'] = round(df_ca_p['ca'].rolling(window=15).mean(), 2)
df_ca_p['moy_mobile_30j'] = round(df_ca_p['ca'].rolling(window=30).mean(), 2)

In [52]:
df_ca

,ca,moy_mobile
date,,
2021-03-31,445918.71,NaN
2021-04-30,439337.85,NaN
2021-05-31,454887.46,446714.67
2021-06-30,447102.17,447109.16
2021-07-31,447593.15,449860.93
2021-08-31,446002.30,446899.21
2021-09-30,470921.80,454839.08
2021-10-31,467397.22,461440.44
2021-11-30,478092.08,472137.03


In [53]:
fig_ca = go.Figure()

fig_ca.add_trace(go.Scatter(
    x=df_ca.index,
    y=df_ca['ca'],
    name="CA",
    mode='lines+markers',
    marker=dict(size=5),
    line=dict(color="#003f5c"),
    hovertemplate="%{y:,.0f} €"
))

fig_ca.add_trace(go.Scatter(
    x=df_ca.index,
    y=df_ca['moy_mobile'],
    name="Moyenne mobile",
    mode='lines+markers',
    marker=dict(size=5),
    line=dict(color='#ffa600'),
    hovertemplate="%{y:,.0f} €"
))

# Habillage
fig_ca.update_layout(
    yaxis=dict(title="Chiffre d'affaire", showgrid=False),
    xaxis=dict(tickformat="%b %Y",),
    legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'),
    height=400,
    width=800,
    hovermode='x unified',
    hoverlabel=dict(font_size=12),
    separators=". ",
    margin=dict(t=50, b=50, l=80, r=40),
)

fig_ca.show()

In [54]:
fig_ca_p = go.Figure()

fig_ca_p.add_trace(go.Scatter(
    x=df_ca_p.index,
    y=df_ca_p['ca'],
    name="CA Journalier",
    mode='lines',
    line=dict(color="#D7D7D7"),
    hovertemplate="%{y:,.0f} €"
))

fig_ca_p.add_trace(go.Scatter(
    x=df_ca_p.index,
    y=df_ca_p['moy_mobile_7j'],
    name="Moyenne 7j",
    mode='lines',
    line=dict(color='#003f5c'),
    hovertemplate="%{y:,.0f} €"
))

fig_ca_p.add_trace(go.Scatter(
    x=df_ca_p.index,
    y=df_ca_p['moy_mobile_30j'],
    name="Moyenne 30j",
    mode='lines',
    line=dict(color='#ffa600'),
    hovertemplate="%{y:,.0f} €"
))

# Habillage
fig_ca_p.update_layout(
    yaxis=dict(title="Chiffre d'affaire", showgrid=False),
    xaxis=dict(tickformat="%d %b %Y"),
    legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'),
    height=400,
    width=1000,
    hovermode='x unified',
    hoverlabel=dict(font_size=12),
    separators=". ",
    margin=dict(t=80, b=50, l=80, r=40),
    title="Evolution du chiffre d'affaire"
)

fig_ca_p.show()

### 1.2 CA par catégorie

In [55]:
# periode

df_categ_p = df_btoc.set_index('date')
df_categ_p = df_categ_p.groupby([pd.Grouper(freq='ME'), 'categ'])['price'].sum().to_frame().reset_index()
df_categ_p = df_categ_p.rename(columns={'price' : 'ca'})
df_categ_p = df_categ_p.pivot(index='date', columns='categ', values='ca')
df_categ_p.head()

# -------------------------

# cumul

df_categ = df_btoc.set_index('date')
df_categ = df_categ.groupby('categ')['price'].sum().reset_index()
df_categ = df_categ.rename(columns={'price' : 'ca'})
total = df_categ['ca'].sum()
df_categ['pct'] = df_categ['ca'] / total * 100
df_categ.head()


,categ,ca,pct
0,0,4119200.69,36.965494
1,1,4520101.86,40.563161
2,2,2504064.46,22.471345


In [56]:
fig_categ = go.Figure()

fig_categ.add_trace(go.Bar(
    name='0',
    x=df_categ_p.index,
    y=df_categ_p[0],
    hovertemplate="%{y:,.0f} €",
    marker_color='#003f5c'
))
fig_categ.add_trace(go.Bar(
    name='1',
    x=df_categ_p.index,
    y=df_categ_p[1],
    hovertemplate="%{y:,.0f} €",
    marker_color='#ffa600'
))
fig_categ.add_trace(go.Bar(
    name='2',
    x=df_categ_p.index,
    y=df_categ_p[2],
    hovertemplate="%{y:,.0f} €",
    marker_color='#bc5090'
))


# Habillage
fig_categ.update_layout(
    yaxis=dict(title="Chiffre d'affaire", showgrid=False),
    xaxis=dict(tickformat="%b %Y",),
    legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'),
    height=400,
    width=1000,
    hovermode='x unified',
    hoverlabel=dict(font_size=12),
    separators=". ",
    margin=dict(t=80, b=50, l=80, r=40),
    title="Evolution du chiffre d'affaire par catégories",
    # barmode='stack'
)

fig_categ.show()

In [57]:
fig_categ = go.Figure()

fig_categ.add_trace(go.Bar(
    name='CA par catégorie',
    x=df_categ.index.astype(str),
    y=df_categ['ca'],
    customdata=df_categ['pct'],
    hovertemplate="%{y:,.0f} € (%{customdata:.1f}%)",
    marker_color=['#1B2A4A', '#0D6E8A', '#E8A020']
))

# Habillage
fig_categ.update_layout(
    yaxis=dict(title="Chiffre d'affaire", showgrid=False),
    legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'),
    height=500,
    width=800,
    hovermode='x unified',
    hoverlabel=dict(font_size=12),
    separators=". ",
    margin=dict(t=50, b=50, l=80, r=40),
)

fig_categ.show()

### 1.3 Métriques clés / mois (clients, transactions, produits)

In [58]:
# Nombre de clients uniques

df_clients = df_btoc.set_index('date')
df_clients = df_clients.groupby(pd.Grouper(freq='MS'))['client_id'].nunique().rename('clients_unique').reset_index()
df_clients.head()

,date,clients_unique
0,2021-03-01,5672
1,2021-04-01,5670
2,2021-05-01,5640
3,2021-06-01,5655
4,2021-07-01,5668


In [59]:
# Nombre de ventes

df_ventes = df_btoc.set_index('date')
df_ventes = df_ventes.groupby(pd.Grouper(freq='MS'))['session_id'].nunique().rename('ventes_mois').reset_index()
df_ventes.head()

,date,ventes_mois
0,2021-03-01,13285
1,2021-04-01,13014
2,2021-05-01,13165
3,2021-06-01,12929
4,2021-07-01,12685


In [60]:
# Nombre de produits vendus

df_produits = df_btoc.set_index('date')
df_produits = df_produits.groupby(pd.Grouper(freq='MS'))['id_prod'].count().rename('produits_vendus').reset_index()
df_produits.head()

,date,produits_vendus
0,2021-03-01,26647
1,2021-04-01,26470
2,2021-05-01,26272
3,2021-06-01,25038
4,2021-07-01,23075


In [61]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.1, 
                    subplot_titles=['Nombre de clients uniques', 'Nombre de ventes', 'Nombre de produits vendus']
                    )

fig.add_trace(go.Scatter(
    x=df_clients['date'],
    y=df_clients['clients_unique'],
    name="Clients Uniques",
    mode='lines+markers',
    marker=dict(size=4),
    line=dict(color='#003f5c'),
    hovertemplate="%{y:,.0f}"
),row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_ventes['date'],
    y=df_ventes['ventes_mois'],
    name="Ventes",
    mode='lines+markers',
    marker=dict(size=4),
    line=dict(color='#bc5090'),
    hovertemplate="%{y:,.0f}"
),row=2, col=1)

fig.add_trace(go.Scatter(
    x=df_produits['date'],
    y=df_produits['produits_vendus'],
    name="Produits vendus",
    mode='lines+markers',
    marker=dict(size=4),
    line=dict(color='#ffa600'),
    hovertemplate="%{y:,.0f}"
),row=3, col=1)

# Habillage
fig.update_layout(
    showlegend=False,
    height=600,
    width=1200,
    hovermode='x unified',
    hoverlabel=dict(font_size=12),
    separators=". ",
    title="Métriques clés / mois"
)

fig.update_xaxes(tickformat="%B %Y", hoverformat="%b %Y")

fig.show()

## 2. Zoom catalogue (produits)

### 2.1 Top / Flop produits

In [62]:
top_flop = df_btoc.groupby('id_prod')['price'].sum().reset_index().sort_values(by='price', ascending=False)
top5 = top_flop.head(5)
flop5 = top_flop.tail(5)

In [63]:
top5

,id_prod,price
3093,2_159,91097.76
3067,2_135,63470.80
3043,2_112,58785.90
3032,2_102,55650.74
2589,1_369,52897.95


In [64]:
flop5

,id_prod,price
718,0_1653,1.98
799,0_1726,1.57
312,0_1284,1.38
594,0_1539,0.99
209,0_1191,0.99


In [65]:
fig = make_subplots(rows=1, cols=2, subplot_titles=['Top 5', 'Flop 5'], horizontal_spacing=0.04)

fig.add_trace(go.Bar(
    name='Top 5',
    x=top5['id_prod'],
    y=top5['price'],
    hovertemplate="%{y:,.0f} €",
    marker_color=['#003f5c', '#58508d', '#bc5090', '#ff6361', '#ffa600']
), row=1, col=1)

fig.add_trace(go.Bar(
    name='Flop 5',
    x=flop5['id_prod'],
    y=flop5['price'],
    hovertemplate="%{y:,.0f} €",
    marker_color=['#003f5c', '#58508d', '#bc5090', '#ff6361', '#ffa600']
), row=1, col=2)

# Habillage
fig.update_layout(
    yaxis=dict(title="Chiffre d'affaire"),
    xaxis=dict(title="ID Produit"),
    xaxis2=dict(title="ID Produit"),
    showlegend=False,
    height=500,
    width=800,
    hovermode='x unified',
    hoverlabel=dict(font_size=12),
    separators=". ",
    margin=dict(t=60, b=25, l=75, r=25),
    title="Top et Flop ventes (CA par produits)"
)

fig.show()

### 2.2 Répartition produits par catégorie

In [66]:
categ = df_btoc.groupby('categ')['id_prod'].count().reset_index()


In [67]:
donuts_categ = go.Figure()

donuts_categ.add_trace(go.Pie(
    labels=categ['categ'],
    values=categ['id_prod'],
    hole=0.65,
    hovertemplate="%{value:,.0f}<extra></extra>",
    marker=dict(colors=['#1B2A4A', '#0D6E8A', '#E8A020']),
    textinfo='label+percent'
))

donuts_categ.update_layout(
    separators=". ",
    width=440,
    margin=dict(t=15, b=15, l=15, r=15),
    showlegend=False
)

## 3. Profils clients

### 3.1 Répartition BtoB vs BtoC

In [68]:
df_donuts = df.groupby('segment')['price'].sum().sort_values(ascending=False).reset_index()
df_donuts

,segment,price
0,BtoC,11143367.01
1,BtoB,884296.09


In [69]:
donuts_btob = go.Figure()

donuts_btob.add_trace(go.Pie(
    labels=df_donuts['segment'],
    values=df_donuts['price'],
    hole=0.65,
    hovertemplate="%{value:,.0f} €<extra></extra>",
    marker=dict(colors=['#0D6E8A', '#E8A020']),
    textinfo='label+percent'
))

donuts_btob.update_layout(
    separators=". ",
    width=440,
    margin=dict(t=15, b=15, l=15, r=15),
    showlegend=False
)

### 3.2 Courbe de Lorenz

In [70]:
ca_par_clients = df_btoc.groupby('client_id')['price'].sum().sort_values(ascending=True).reset_index()

# Lorenz
ca = ca_par_clients.query("price > 0")
ca_sorted = np.sort(ca.price)

lorenz = np.cumsum(ca_sorted) / ca_sorted.sum()
lorenz = np.insert(lorenz, 0, 0)

xaxis = np.linspace(0, 1, len(lorenz))

# Gini
auc = np.trapezoid(lorenz, xaxis)
gini = 1 - 2 * auc

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=xaxis,
    y=lorenz,
    line=dict(color='#0D6E8A'),
    hovertemplate="%{x:.0%} des clients → %{y:.0%} du CA<extra></extra>"    
))

fig.add_trace(go.Scatter(
    x=[0,1],
    y=[0,1],
    line=dict(dash='dash', color='#E8A020')
))

# Habillage
fig.update_layout(
    xaxis=dict(title="Part cumulée des clients"),
    yaxis=dict(title="Part cumulée du CA"),
    showlegend=False,
    height=600,
    width=800,
    hoverlabel=dict(font_size=12),
    separators=". ",
    margin=dict(t=50, b=50, l=80, r=40)
)

fig.show()

## KPIs

### CA

In [71]:
# CA Total
ca_total = df.price.sum()
ca_total

np.float64(12027663.100000001)

In [72]:
# CA BtoC
ca_btoc = df_btoc.price.sum()
ca_btoc

np.float64(11143367.01)

In [73]:
# CA BtoB
ca_btob = ca_total - ca_btoc
ca_btob

np.float64(884296.0900000017)

### Clients & Catalogue

In [74]:
# Moy clients uniques
moy_unique = df_clients.clients_unique.mean()
moy_unique

np.float64(5755.375)

In [75]:
# Moy ventes (sessions)
moy_ventes = df_ventes.ventes_mois.mean()
moy_ventes

np.float64(13437.958333333334)

In [76]:
# Catalogue
catalogue = df.id_prod.nunique()
catalogue

3265

In [77]:
# Best Categ
best_categ = df_btoc.groupby('categ')['price'].sum()
best_categ

categ
0    4119200.69
1    4520101.86
2    2504064.46
Name: price, dtype: float64

In [78]:
# Best PDT
best_pdt = df_btoc.groupby('id_prod')['price'].sum().sort_values(ascending=False)
best_pdt.head(1)

id_prod
2_159    91097.76
Name: price, dtype: float64

## Complements d'analyse

In [79]:
# prix moyen des categ (pour determiner si cat 2 = premium)
prix_moyen_categ = df_btoc.groupby('categ')['price'].mean()
print(prix_moyen_categ)

categ
0    10.636207
1    20.489571
2    76.231870
Name: price, dtype: float64


## Export

In [80]:
df_btoc.to_csv(r"..\data\processed\btoc.csv")